# Notebook 3 — Adding sectors & shocks: the AI scenarios

This is where input-output analysis becomes a **policy/foresight tool**. Starting
from the compact, real-data model we built and saved in Notebook 2 (7 sectors ×
6 macro-regions), we will:

- **A.** run a simple, fully transparent **final-demand shock** on *Electrical
  equipment* — the "hello world" of scenario analysis;
- **B.** **add a new sector** that does not exist in official statistics yet —
  *AI Hyperscalers*, located only in the USA, with a GPU- and electricity-hungry
  cost structure, selling a *token-generation service*;
- **C.** build the headline scenario: **European service firms replace part of
  their high-skill labour with AI tokens**;
- **D.** read off the **winners and losers** with a few visualizations.

> 🧩 Everything downstream of "real EXIOBASE numbers" is illustrative: the AI cost
> structure and the substitution intensity are **plausible assumptions you can
> calibrate**, not official figures. The point is the *method*.

In [32]:
import mario
import os
import numpy as np
import pandas as pd

mario.set_log_verbosity("warning")
print("MARIO version:", mario.__version__)

# Path to the parquet snapshot written by Notebook 2.
PARQUET_PATH = os.path.join("outputs", "exio_aggregated_parquet")
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

MARIO version: 1.0.2


## Load the model saved by Notebook 2

We round-trip the parquet snapshot instead of re-parsing the multi-GB raw
EXIOBASE — fast and reproducible.

In [59]:
db = mario.parse_from_parquet(
    PARQUET_PATH,
    table="IOT",
    mode="flows",
    new_scenario="baseline",
    flat=True,
)
db.calc_all(["X", "Z", "z", "Y", "v", "e"])
print(db)
print("\nSectors:", db.sectors)
print("Regions:", db.regions)
print("Value-added factors:", db.get_index("Factor of production"))
print("Satellite accounts:", db.get_index("Satellite account"))

name = None
table = IOT
scenarios = ['baseline']
Factor of production = 9
Satellite account = 2
Consumption category = 7
Region = 6
Sector = 7


Sectors: ['Agriculture', 'Electrical equipment', 'Manufacturing', 'Non-renewable electricity', 'Rare materials', 'Renewable electricity', 'Services']
Regions: ['Africa', 'China', 'Europe', 'Rest of Asia', 'Rest of the World', 'USA']
Value-added factors: ['Taxes less subsidies on products purchased: Total', 'Other net taxes on production', "Compensation of employees; wages, salaries, & employers' social contributions: Low-skilled", "Compensation of employees; wages, salaries, & employers' social contributions: Medium-skilled", "Compensation of employees; wages, salaries, & employers' social contributions: High-skilled", 'Operating surplus: Consumption of fixed capital', 'Operating surplus: Rents on land', 'Operating surplus: Royalties on resources', 'Operating surplus: Remaining net operating surplus']
Satellite accounts: ['Water Consumption', 

## Part A — A simple final-demand shock

**Question:** what happens to the whole system if European households and firms
demand **+30% of Electrical equipment**?

Because we kept *Electrical equipment* as an explicit sector in Notebook 2, we can
shock it directly. MARIO's shock workflow is:

1. generate a shock template with `get_shock_excel(...)`;
2. fill the relevant sheet — here **`Y`** (final demand) — with the rows to shock;
3. apply it with `shock_calc(...)`, which creates a **new scenario** and leaves
   `baseline` untouched.

The `Y` sheet columns are `Region_from, Sector_from, Region_to,
Consumption category_to, type, value`. With `type="Percentage"`, MARIO applies
`Y_new = Y_old × (1 + value)`, so `value=0.30` means +30%.

In [3]:
SHOCK_PATH = os.path.join(OUT_DIR, "shock_1.xlsx")
db.get_shock_excel(path=SHOCK_PATH)


**Fill in the shock.** Open `outputs/shock_1.xlsx`, go to the **`Y`** sheet
and write a single row (use the consumption-category label printed above):

| Region_from | Sector_from | Region_to | Consumption category_to | type | value |
|---|---|---|---|---|---|
| Europe | Electrical equipment | Europe | Final consumption expenditure by households | Percentage | 0.30 |

Save the file, then apply it below.

In [34]:
SHOCK_PATH = os.path.join(OUT_DIR, "shock_1.xlsx")
db.shock_calc(SHOCK_PATH, Y=True, scenario="Europeans buy more GPU")
print("Scenarios now available:", db.scenarios)

Scenarios now available: ['baseline', 'Europeans buy more GPU']


### Read the effect

Compare total output by sector, baseline vs shock. The increase propagates **up
the supply chain**: not only Electrical equipment, but also the electricity, rare
materials and manufacturing that feed into it.

In [35]:
db.query('Y', 
         scenarios=["baseline","Europeans buy more GPU"], 
         base_scenario="baseline")["Europeans buy more GPU"].loc[:,('Europe',slice(None),'Final consumption expenditure by households')]

Region                                                                                  Europe
Level                                                                     Consumption category
Item                                               Final consumption expenditure by households
Region            Level  Item                                                                 
Africa            Sector Agriculture                                                  0.000000
                         Electrical equipment                                         0.000000
                         Manufacturing                                                0.000000
                         Non-renewable electricity                                    0.000000
                         Rare materials                                               0.000000
                         Renewable electricity                                        0.000000
                         Services                                                     0.000000
China             Sector Agriculture                                                  0.000000
                         Electrical equipment                                         0.000000
                         Manufacturing                                                0.000000
                         Non-renewable electricity                                    0.000000
                         Rare materials                                               0.000000
                         Renewable electricity                                        0.000000
                         Services                                                     0.000000
Europe            Sector Agriculture                                                  0.000000
                         Electrical equipment                                      3565.028041
                         Manufacturing                                                0.000000
                         Non-renewable electricity                                    0.000000
                         Rare materials                                               0.000000
                         Renewable electricity                                        0.000000
                         Services                                                     0.000000
Rest of Asia      Sector Agriculture                                                  0.000000
                         Electrical equipment                                         0.000000
                         Manufacturing                                                0.000000
                         Non-renewable electricity                                    0.000000
                         Rare materials                                               0.000000
                         Renewable electricity                                        0.000000
                         Services                                                     0.000000
Rest of the World Sector Agriculture                                                  0.000000
                         Electrical equipment                                         0.000000
                         Manufacturing                                                0.000000
                         Non-renewable electricity                                    0.000000
                         Rare materials                                               0.000000
                         Renewable electricity                                        0.000000
                         Services                                                     0.000000
USA               Sector Agriculture                                                  0.000000
                         Electrical equipment                                         0.000000
                         Manufacturing                                                0.000000
          

Let's see which sectors are the most impacted.

In [36]:
df = db.query(
    'X',
    scenarios=["baseline", "Europeans buy more GPU"],
    base_scenario="baseline"
)["Europeans buy more GPU"]

df.sort_values(df.columns[0], ascending=False).head(10)

production
Region       Level  Item                             
Europe       Sector Electrical equipment  4040.174721
                    Services              2492.003459
                    Manufacturing          892.111344
Rest of Asia Sector Manufacturing          102.185683
                    Services                96.091209
Europe       Sector Rare materials          83.943004
USA          Sector Services                80.382001
China        Sector Manufacturing           79.300380
                    Electrical equipment    78.054585
USA          Sector Manufacturing           42.422407

In [8]:
fig = db.plot(
    matrix="X",
    scenarios=["Europeans buy more GPU"],
    x = "Item",
    color = "Region",
    base_scenario="baseline",
    difference="absolute",
    path=False
)
fig

## Part B — Add a new sector: *AI Hyperscalers*

Official statistics don't have an "AI compute" sector yet. With `add_sectors(...)`
we can **inject one** by describing its cost structure (its column of the
input-output table) plus its total output and where that output goes.

### B.1 The cost structure of an AI hyperscaler

We describe *AI Hyperscalers* (USA) as an **inventory**: how much of each input it
needs to produce **one unit (1 M€) of token service**. In a monetary IOT the
intermediate-input shares plus the value-added share must sum to **1** (output =
costs + value added). Our assumptions:

| Input | Source region | Share of 1 M€ output |
|---|---|---|
| Electrical equipment (servers, GPUs) | USA | 0.18 |
| Electrical equipment (chips) | Rest of Asia | 0.12 |
| Non-renewable electricity | USA | 0.10 |
| Renewable electricity | USA | 0.08 |
| Rare materials | China | 0.04 |
| Manufacturing (cooling, construction) | USA | 0.05 |
| Services (real estate, telecom, O&M) | USA | 0.13 |
| **Value added** (capital + labour) | USA | **0.30** |
| **Total** | | **1.00** |

On top of the monetary structure we attach **direct environmental flows**: CO₂
(backup/own generation) and a large **Water** demand for cooling. These are
satellite accounts, so they sit *outside* the monetary balance. Their levels here
are illustrative placeholders — calibrate them to your favourite source.

### B.2 Size the new sector and the token price

We give the sector a baseline output level and a **token price** so we can talk in
tokens rather than euros. With a price of **$10 per 1 million tokens**, a
50 000 M€ (~\$54 bn) sector corresponds to a few thousand *trillion* tokens a
year — the right order of magnitude for today's frontier hyperscalers.

In the baseline, the whole output is sold to final demand (no sector buys AI
tokens yet — that is exactly what changes in Part C).

In [44]:
ADD_SECT_PATH = r"C:\Users\nicog\GitHub\deep-dive\notebooks\outputs\AI.xlsx"
db.get_add_sectors_excel(ADD_SECT_PATH)

WrongInput: Add-sectors workbook 'C:\Users\nicog\GitHub\deep-dive\notebooks\outputs\AI.xlsx' already exists. Pass overwrite=True to replace it.

In [47]:
db.read_add_sectors_excel(
    path=r"C:\Users\nicog\GitHub\deep-dive\notebooks\outputs\AI.xlsx",
    get_inventories=True,
)

### B.3 Generate the template and fill the recipe

`get_add_sectors_excel(...)` pre-fills the `Region`, `Sector`, `Inventory sheet`
and `Add or Split` columns for us; we complete the rest by hand.

Open `outputs/add_ai_sector.xlsx` and fill two sheets.

**`Master` sheet** — complete the (already started) row for *AI Hyperscalers* /
*USA*:

| Quantity | Unit | Final consumption | Consumption category |
|---|---|---|---|
| 50000 | *(Sector unit, e.g. `M.EUR`)* | 50000 | *(your consumption category)* |

(`Region`, `Sector`, `Inventory sheet` = `INV_001`, and `Add or Split` = `Add` are
pre-filled.)

**`INV_001` sheet** — the production recipe: how much of each input per **1 unit
(1 M€)** of token service. The `Sector` + `Factor of production` quantities are
shares that **sum to 1** (output = costs + value added); here the 0.30 of value
added is split across two factors (fixed capital + high-skill labour). The two
satellite rows are direct environmental flows (their magnitudes are illustrative
placeholders to calibrate):

| Quantity | Unit | Input | Item type | DB Item | DB Region | Change type |
|---|---|---|---|---|---|---|
| 0.18 | M.EUR | GPU/servers | Sector | Electrical equipment | USA | Update |
| 0.12 | M.EUR | AI chips | Sector | Electrical equipment | Rest of Asia | Update |
| 0.10 | M.EUR | grid power | Sector | Non-renewable electricity | USA | Update |
| 0.08 | M.EUR | green power | Sector | Renewable electricity | USA | Update |
| 0.04 | M.EUR | rare metals | Sector | Rare materials | China | Update |
| 0.05 | M.EUR | cooling/build | Sector | Manufacturing | USA | Update |
| 0.13 | M.EUR | real estate/O&M | Sector | Services | USA | Update |
| 0.18 | M.EUR | capital+labour | Factor of production | Operating surplus: Consumption of fixed capital | USA | Update |
| 0.12 | M.EUR | capital+labour | Factor of production | Compensation of employees; wages, salaries, & employers' social contributions: High-skilled | USA | Update |
| 400000 | kg | direct CO2 | Satellite account | CO2 | USA | Update |
| 0.003 | Mm3 | cooling H₂O | Satellite account | Water Consumption | USA | Update |

Save the file, then apply it below.

In [64]:
ADD_PATH = r"C:\Users\nicog\GitHub\deep-dive\notebooks\outputs\AI_filled.xlsx"

db.read_add_sectors_excel(ADD_PATH, read_inventories=True)
db_ex = db.add_sectors(io=ADD_PATH, inplace=False)



WARNING Database: All scenarios will be deleted from the database after add_sectors.


In [65]:
db_ex.sectors

['AI Hyperscalers',
 'Agriculture',
 'Electrical equipment',
 'Manufacturing',
 'Non-renewable electricity',
 'Rare materials',
 'Renewable electricity',
 'Services']

In [66]:
db_ex.z.loc[:,("USA",slice(None),"AI Hyperscalers")]

Region                                                         USA
Level                                                       Sector
Item                                               AI Hyperscalers
Region            Level  Item                                     
Africa            Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.00
                         Manufacturing                        0.00
                         Non-renewable electricity            0.00
                         Rare materials                       0.00
                         Renewable electricity                0.00
                         Services                             0.00
China             Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.00
                         Manufacturing                        0.00
                         Non-renewable electricity            0.00
                         Rare materials                       0.04
                         Renewable electricity                0.00
                         Services                             0.00
Europe            Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.00
                         Manufacturing                        0.00
                         Non-renewable electricity            0.00
                         Rare materials                       0.00
                         Renewable electricity                0.00
                         Services                             0.00
Rest of Asia      Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.12
                         Manufacturing                        0.00
                         Non-renewable electricity            0.00
                         Rare materials                       0.00
                         Renewable electricity                0.00
                         Services                             0.00
Rest of the World Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.00
                         Manufacturing                        0.00
                         Non-renewable electricity            0.00
                         Rare materials                       0.00
                         Renewable electricity                0.00
                         Services                             0.00
USA               Sector AI Hyperscalers                      0.00
                         Agriculture                          0.00
                         Electrical equipment                 0.18
                         Manufacturing                        0.05
                         Non-renewable electricity            0.10
                         Rare materials                       0.00
                         Renewable electricity                0.08
                         Services                             0.13

## Part C — Scenario: EU services swap high-skill labour for AI tokens

The headline question. We assume European **Services** firms reroute a slice of
their spending: instead of paying for **high-skill labour** (a value-added
component), they **buy AI token services from the USA**.

We implement this as a balanced shift in the *Services / Europe* cost column:

- **subtract** an amount `Δ` from the labour value-added coefficient;
- **add** the same `Δ` as an intermediate purchase from *AI Hyperscalers (USA)*.

Because we move `Δ` from value added to intermediate inputs, the column still sums
to 1 — the economy stays balanced. We size `Δ` as a fraction of the current
labour coefficient.

In [67]:
SUB_PATH = os.path.join(OUT_DIR, "shock_ai_substitution.xlsx")
db_ex.get_shock_excel(path=SUB_PATH, num_shock=2)
print("Empty shock template written to:", SUB_PATH)

Empty shock template written to: outputs\shock_ai_substitution.xlsx


In [ ]:
db.V.loc[
    "Compensation of employees; wages, salaries, & employers' social contributions: High-skilled",
    ('Europe', slice(None), 'Services')
]

Region  Level   Item    
Europe  Sector  Services    4.801966e+06
Name: Compensation of employees; wages, salaries, & employers' social contributions: High-skilled, dtype: float64

In [72]:
db.v.loc[
    "Compensation of employees; wages, salaries, & employers' social contributions: High-skilled",
    ('Europe', slice(None), 'Services')
]

Region  Level   Item    
Europe  Sector  Services    0.200944
Name: Compensation of employees; wages, salaries, & employers' social contributions: High-skilled, dtype: float64

Open `outputs/shock_ai_substitution.xlsx` and fill **two** sheets, using the
numbers printed above (`type = Update` sets the coefficient to the given value):

**`v` sheet** — cut the labour coefficient of EU Services to `labour − delta`:

| Factor of production_from | Region_to | Sector_to | type | value |
|---|---|---|---|---|
| Compensation of employees; wages, salaries, & employers' social contributions: High-skilled | Europe | Services | Update | *(new labour coeff., printed above)* |

**`z` sheet** — add an AI-tokens input into EU Services equal to `delta`:

| Region_from | Sector_from | Region_to | Sector_to | type | value |
|---|---|---|---|---|---|
| USA | AI Hyperscalers | Europe | Services | Update | *(delta, printed above)* |

Moving `delta` from value added to intermediate inputs keeps the column summed to
1 — the economy stays balanced. Save, then apply below.

In [75]:
db_ex.shock_calc(SUB_PATH, z=True, v=True, scenario="ai_substitution")
print("Scenarios:", db_ex.scenarios)

WrongInput: Scenario ai_substitution already exist. In order to re-write the scenario, you can use force_rewrite = True.

## Part D — Winners and losers

Now we read the scenario. The substitution does two things at once:

- it **creates demand** for AI compute (USA) and everything that feeds it
  (electrical equipment, electricity, rare materials) — the **winners**;
- it **shrinks** high-skill labour income in Europe and can ripple into the
  sectors that depended on it — the **losers**.

We compare `ai_substitution` against `baseline` along three dimensions: sectoral
output, regional GDP, and CO₂.

In [80]:
fig = db_ex.plot(
    matrix="X",
    scenarios=["ai_substitution"],
    x = "Item",
    color = "Region",
    base_scenario="baseline",
    difference="absolute",
    path=False
)
fig

In [78]:
fig = db_ex.plot(
    matrix="V",
    scenarios=["ai_substitution"],
    base_scenario="baseline",
    difference="absolute",
    path=False
)
fig

### Reading the result

Typical story this scenario tells (your exact numbers depend on the calibration):

- **Winners:** *AI Hyperscalers* (USA, by construction), and its supply chain —
  *Electrical equipment*, *Non-renewable* & *Renewable electricity*, *Rare
  materials*. The gains concentrate in the **USA** and in **Rest of Asia / China**
  (chips and metals).
- **Losers:** **European high-skill labour income** falls (value added rerouted to
  imported AI services), and Europe's GDP share slips even if its Services output
  holds up — value that used to be paid to European workers now flows to US
  compute providers.
- **Emissions:** CO₂ tends to **relocate to the USA**, where the electricity- and
  hardware-intensive compute is produced.

### 🧩 Experiment

- Raise `SUBSTITUTION_SHARE` to 0.30 — does Europe's GDP loss scale linearly?
- Make the AI sector greener (shift its electricity mix toward *Renewable
  electricity*) and re-run: how much does the CO₂ relocation shrink?
- Add a second hyperscaler in *Europe* and compare a "sovereign AI" path.

That's the full arc — from raw tables, to a real aggregated model, to a
forward-looking AI scenario. 🎓